# Tutorial: how to work with PostgreSQL

By Efraim Safrai , 2026-01


## Steps to install and connect to PostgreSQL


first click on the link [here](https://www.enterprisedb.com/downloads/postgres-postgresql-downloads) choose and download the latest version that suits your computer

while installing press next and when you have to create password remmember it because we will use it later

also make sure in the port window you have the number 5432
<img src="res/pg_tutorial/tutorial_pic1.png" alt="tutorial_pic1.png" style="width: 500px; display: block; margin: 0 auto;">

after you finished open PyCharm and we will connect to the database

To connect to your postgresql db you should follow the next steps:
1. click on the db icon (on the top right corner)
2. click on the + (on the top left corner)
3. click on Data source
4. select PostgreSQL


 the steps in the picture
 <img src="res/pg_tutorial/tutorial_pic2.png" alt="tutorial_pic2.png" style="width: 500px; display: block; margin: 0 auto;">

in the next window you will have to connect to your account.
first of all if you have a notification on the bottom of your  window that says "Download missing driver files" download them.

then make sure that:


*  Host is localhost
*   Port is 5432
* User is postgres
* password is the password you chose in the installing phase
* the Database is postgers

after that press ok

<img src="res/pg_tutorial/tutorial_pic3.PNG" alt="tutorial_pic3.PNG" style="width: 500px; display: block; margin: 0 auto;">

## Connecting to a DB running on host machine, when the notebook is in Docker

The docker client has a different IP and is considered a different machine, so the DB server has to listen to all addresses (or at least the one belonging the the container).

These instructions are for Linux machine. Adapt as needed.

edit `/etc/postgresql/16/main/postgresql.conf` <br>
find `#listen_addresses = 'localhost'` <br>
and replace with <br>
`listen_addresses = '*'` <br>

And in `/etc/postgresql/16/main/pg_hba.conf` add line to allow docker (or any) network:
```
 TYPE  DATABASE        USER            ADDRESS                 METHOD
 host    all             all             0.0.0.0/0               scram-sha-256
```
restart the service: `sudo systemctl restart postgresql`

> NOTE: rememeber that now the DB server is open to the whole world!

In the container, the JDBC URL has to be updated with the DB address
<hr>


now you have connected to the database.
you can add tables under postgres ->public->tables.
you can drag and drop data files in there or right click -> New to choose other options(DDL etc.)

> NOTE: for your convenience, the data file is in the /data folder

<img src="res/pg_tutorial/tutorial_pic4.PNG" alt="tutorial_pic4.PNG" style="width: 500px; display: block; margin: 0 auto;">


In [2]:
host_ip = "172.17.0.1" # common value of Docker network. 
pg_jar_name = "postgresql-42.7.2.jar" # update this according to the actual version you use

## Reading from the DB with spark




In [3]:
import pyspark
from pyspark.sql import SparkSession
import pyspark.sql.functions
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window



In [4]:

spark = SparkSession.builder \
    .appName("PostgresJDBC_ParallelRead") \
    .config("spark.jars", f"/home/jars/{pg_jar_name}") \
    .config("spark.driver.extraClassPath", pg_jar_name) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

for more efficient parallel read we will use JDBC driver to read from postgresql while partitioning the data based on the Incident Zip column.

In [5]:
jdbc_url = f"jdbc:postgresql://{host_ip}:5432/postgres"

properties = {
    "user": "postgres",           # Your PostgreSQL username, in this tutotial we used 'postgres'
    "password": "noam",  # enter your PostgreSQL password that you created during installation
    "driver": "org.postgresql.Driver"
}

In [6]:
# we will cast the Incident Zip column to integer in order to use it for partitioning
query = """(
    SELECT
        "Building Type",
        CAST("Incident Zip" AS INTEGER) as incident_zip,
        "borough",
        "Day of Week",
        "Date of Report",
        "Duration of Call (min)"
    FROM public.noisy_neighbors_reports
) as subquery"""

df = spark.read.jdbc(
    url=jdbc_url,
    table=query,                # Pass query instead of table name
    column="incident_zip",      # Use the casted column name
    lowerBound=10001,
    upperBound=11255,
    numPartitions=10,
    properties=properties
)

df.show(5)


+-----------------+------------+---------+-----------+--------------+----------------------+
|    Building Type|incident_zip|  borough|Day of Week|Date of Report|Duration of Call (min)|
+-----------------+------------+---------+-----------+--------------+----------------------+
|            Hotel|       10068|Manhattan|          3|    2023-03-22|                     8|
|Apartment Complex|       10061|Manhattan|          7|    2023-02-26|                    52|
|            Hotel|       10050|Manhattan|          1|    2023-12-18|                    31|
|            Hotel|       10093|Manhattan|          5|    2023-06-16|                    14|
|Apartment Complex|       10090|Manhattan|          4|    2023-03-16|                     2|
+-----------------+------------+---------+-----------+--------------+----------------------+
only showing top 5 rows



we can see that the schema is not correct so we will cast the columns to the right types

In [7]:
df.printSchema()

root
 |-- Building Type: string (nullable = true)
 |-- incident_zip: integer (nullable = true)
 |-- borough: string (nullable = true)
 |-- Day of Week: string (nullable = true)
 |-- Date of Report: string (nullable = true)
 |-- Duration of Call (min): string (nullable = true)



In [8]:
df = df.select(
    col("Building Type").cast(StringType()),
    col("incident_zip").cast(DoubleType()).alias("Incident Zip"),
    col("borough").cast(StringType()).alias("Borough"),
    col("Day of Week").cast(IntegerType()),
    col("Date of Report").cast(StringType()),
    col("Duration of Call (min)").cast(IntegerType())
)

In [9]:
df.printSchema()

root
 |-- Building Type: string (nullable = true)
 |-- Incident Zip: double (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Day of Week: integer (nullable = true)
 |-- Date of Report: string (nullable = true)
 |-- Duration of Call (min): integer (nullable = true)



## Queries

### Query 1:
 which day of the week has the highest number of calls?

In [10]:
#order by total calls per day of week
query1_result = df \
    .groupBy(col("Day of Week").alias("day_of_week")) \
    .agg(count("*").alias("total_calls")) \
    .orderBy(col("total_calls").desc())

# Get the day with highest number of calls
highest_day = query1_result.first()
print(f" Day {highest_day['day_of_week']} has the most calls ({highest_day['total_calls']:,} calls)")


 Day 7 has the most calls (10,022 calls)


## Query 2:

now we would like to mesure the police efficiency for the common citizens in each borough and return the top 5 borough with the lowst average call duration.
 we will assume for this task that the common citizens lives only in  private homes and apartment complexes

In [11]:
query2_result = df \
    .filter((col("Building Type") == "Private Home") | (col("Building Type") == "Apartment Complex") )\
    .groupBy("borough") \
    .agg(avg("Duration of Call (min)").alias("avg_duration")) \
    .orderBy(col("avg_duration"))
query2_result.show(5)

+-------------+------------------+
|      borough|      avg_duration|
+-------------+------------------+
|       Queens|30.397954579636036|
|Staten Island| 30.43998208420424|
|    Manhattan|30.502072232089994|
|     Brooklyn|30.549180327868854|
|        Bronx|30.650838825644097|
+-------------+------------------+



### Query 3:
now we would like to find for each borough what is the building type that got the highest duration call time

in order to solve this query we will use window functions

In [12]:
win_bor = Window.partitionBy("borough").orderBy(col("Duration of Call (min)").desc() )

df.withColumn("row_number",row_number().over(win_bor)) \
  .where(col("row_number")==1) \
  .select("borough","Building Type") \
  .show()


+-------------+-----------------+
|      borough|    Building Type|
+-------------+-----------------+
|        Bronx|     Private Home|
|     Brooklyn|            Hotel|
|    Manhattan|            Hotel|
|       Queens|Apartment Complex|
|Staten Island|     Private Home|
+-------------+-----------------+

